In [12]:
# Instalacion de dependencias
# Click una sola ves

import subprocess
import sys
import hashlib #libreria
import hmac #libreria
import os


a. SHA-256
1. Aplica hashlib para obtener el hash de textos y archivos, muestra cómo cambia si el contenido se modifica.
2. Explica en el video la importancia de la irreversibilidad y resistencia a colisiones.

In [13]:
def hash_text(text):
    """
    Calcula el hash SHA-256 de un texto.
    """
    return hashlib.sha256(text.encode()).hexdigest()

def hash_file(filename):
    """
    Calcula el hash SHA-256 de un archivo.
    Lee el archivo en bloques para no cargarlo todo en memoria.
    """
    sha256 = hashlib.sha256()
    with open(filename, "rb") as f:
        for block in iter(lambda: f.read(4096), b""):  # Lee en bloques de 4 KB
            sha256.update(block)
    return sha256.hexdigest()

b. Detección de integridad
1. Crea función para comparar hashes de dos archivos y verificar si son idénticos.
2. Muestra y explica un ejemplo en el video.

In [ ]:
def compare_files(file1, file2):
    """
    Compara los hashes SHA-256 de dos archivos.
    Retorna True si son idénticos, False en caso contrario.
    """
    hash1 = hash_file(file1)
    hash2 = hash_file(file2)
    return hash1 == hash2

c. HMAC
1. Investiga y programa la generación y verificación de HMAC con SHA-256 y clave.
2. Comenta en el video cuándo usar HMAC y diferencias con hash simple.

In [23]:
def generate_hmac(message, key):
    """
    Genera un HMAC-SHA256 a partir de un mensaje y una clave.
    """
    return hmac.new(key.encode(), message.encode(), hashlib.sha256).hexdigest()

def verify_hmac(message, key, hmac_to_verify):
    """
    Verifica si un HMAC recibido coincide con el generado.
    """
    expected_hmac = generate_hmac(message, key)
    return hmac.compare_digest(expected_hmac, hmac_to_verify)

Autenticación con hash y sal 
Simula registro/inicio de sesión guardando el hash (usa SHA-256).
Integra el concepto de “sal” para evitar ataques de diccionario.
Explica riesgos y la importancia de la sal en el video

In [24]:
def register_usuario(password):
    """
     Registro de un usuario:
    - Genera una sal aleatoria (16 bytes)
    - Calcula hash de la contraseña + sal
    - Retorna sal y hash para guardar en la base de datos
    """
    salt = os.urandom(16).hex()  # Se guarda como string hexadecimal
    salted_pass = password + salt
    hashed_pass = hashlib.sha256(salted_pass.encode()).hexdigest()
    return salt, hashed_pass

def login_usuario(password, salt, stored_hash):
    """
     Inicio de sesión:
    - Usa la misma sal que se generó en el registro
    - Recalcula el hash y lo compara con el guardado
    """
    salted_pass = password + salt
    hashed_pass = hashlib.sha256(salted_pass.encode()).hexdigest()
    return hashed_pass == stored_hash

Pruebas Proyecto

In [27]:
print("\n===== HASH DE TEXTO =====")
text1 = "Sabado sin pola"
text2 = "sabado sin pola!"
print("Texto 1:", hash_text(text1))
print("Texto 2:", hash_text(text2))
print("Nota: el hash cambia por completo por un !.\n")

print("===== HASH DE ARCHIVOS =====")
#Para probar esto se creo 4 archivos de texto:
# Archivo1.txt con "Sabados con s"
# Archivo2.txt con "Sabados con S ! !"
# Archivo3.txt con "Sabados sin s"
# Archivo4.txt con "Sabados con s !"
# Se crea un Archivo4 con un mensaje para comparar 
file1 = "Archivo1.txt"
file2 = "Archivo2.txt"
file3 = "Archivo3.txt "
print(f"Hash {file1}: {hash_file(file1)}")
print(f"Hash {file2}: {hash_file(file2)}")
print(f"Hash {file3}: {hash_file(file3)}")

print("\n¿Los archivos son idénticos?:", compare_files(file1, file2))
print("\nAhora se comprueba con un mensaje correcto")
print("\n¿Los archivos son idénticos?:", compare_files(file1, file3))
print("\n===== HMAC =====")
key = "clave_secreta"
msg = "Mensaje importante"
hmac_value = generate_hmac(msg, key)
print("HMAC generado:", hmac_value)
print("Verificación correcta:", verify_hmac(msg, key, hmac_value))
print("Se verifica ahora con otra_clave")
print("Verificación incorrecta:", verify_hmac(msg, "otra_clave", hmac_value))

print("\n===== AUTENTICACIÓN CON SAL =====")
# Registro
user_password = "123456"
salt, stored_hash = register_usuario(user_password)
print("Sal guardada:", salt)
print("Hash guardado:", stored_hash)

# Inicio de sesión correcto
print("Inicio de sesión (correcto):", login_usuario("123456", salt, stored_hash))
print("Se prueba ahora con otro mensaje distinto")
# Inicio de sesión incorrecto
print("Inicio de sesión (incorrecto):", login_usuario("password_mal", salt, stored_hash))


===== HASH DE TEXTO =====
Texto 1: 1043b31fc155a10124ee6ce52e81c88e7f9627081d136a61da6ba3dd39f539f2
Texto 2: 187c20c44407c38fa0f386bf447cffa9a7a61fa2f592ce6a013292e71be380a5
Nota: el hash cambia por completo por un !.

===== HASH DE ARCHIVOS =====
Hash Archivo1.txt: 505863000172ef9d57393de1133cefe7cd03cba62d1fd066929767b2899cfc9e
Hash Archivo2.txt: 76eee6b8f61cc0fc7fc258433a2a593bffc869718ae17e2c6646260acb201734
Hash Archivo3.txt : 505863000172ef9d57393de1133cefe7cd03cba62d1fd066929767b2899cfc9e

¿Los archivos son idénticos?: False

Ahora se comprueba con un mensaje correcto

¿Los archivos son idénticos?: True

===== HMAC =====
HMAC generado: 220f0c7697dc266be77e2f9797428adcab7604ad598dc2cbf070d7a018f619d9
Verificación correcta: True
Se verifica ahora con otra_clave
Verificación incorrecta: False

===== AUTENTICACIÓN CON SAL =====
Sal guardada: 420af2ad4d72d563c27d840a3c01d4e4
Hash guardado: 86b845a065b6c146f7e4e910a44dc0c1c610828e138b48b042b24294a2a9c4a8
Inicio de sesión (correcto): 